# 2D Gaussian Splatting — represent an image with Gaussians

**Track B · 3D & Neural Rendering** · maps to lesson B7 (3D Gaussian Splatting).

3D Gaussian Splatting represents a scene as millions of coloured Gaussians, rendered differentiably and optimized to match photos. Here is the same idea in **2D**: optimize `N` anisotropic Gaussians (position · scale · rotation · colour · opacity) so their splat reconstructs a target image. Everything (including the gradient through the renderer) is plain PyTorch.

> CPU works; a GPU lets you raise `N` for crisp detail.

In [ ]:
import os, math, torch, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 800)); N = int(os.environ.get("NGAUSS", 500)); H = W = 64
print("device:", device, "| steps:", STEPS, "| gaussians:", N)

## 1 · Target image (procedural, self-contained)

In [ ]:
ys = torch.linspace(0, 1, H, device=device); xs = torch.linspace(0, 1, W, device=device)
YY, XX = torch.meshgrid(ys, xs, indexing="ij"); coords = torch.stack([XX, YY], -1)   # (H,W,2)
target = 0.15 + 0.2 * YY[..., None].repeat(1, 1, 3)                                   # soft gradient
for cx, cy, r, col in [(.3,.35,.16,(.95,.35,.35)), (.68,.4,.2,(.25,.5,.95)), (.5,.72,.17,(.35,.85,.45))]:
    g = torch.exp(-(((XX-cx)**2 + (YY-cy)**2) / (2*r*r)))
    target = target + g[..., None] * torch.tensor(col, device=device)
target = target.clamp(0, 1)
plt.imshow(target.cpu()); plt.title("target"); plt.axis("off"); plt.show()

## 2 · The Gaussians — learnable position, scale, rotation, colour, opacity

In [ ]:
pos  = torch.rand(N, 2, device=device, requires_grad=True)
logs = torch.full((N, 2), math.log(0.06), device=device, requires_grad=True)
rot  = torch.zeros(N, device=device, requires_grad=True)
col  = torch.randn(N, 3, device=device, requires_grad=True)
op   = torch.zeros(N, device=device, requires_grad=True)

def render():
    s = torch.exp(logs); c, si = torch.cos(rot), torch.sin(rot)
    d = coords[:, :, None, :] - pos[None, None, :, :]            # (H,W,N,2)
    dx = d[..., 0] * c + d[..., 1] * si
    dy = -d[..., 0] * si + d[..., 1] * c                         # delta in each Gaussian's frame
    e = torch.exp(-0.5 * ((dx / s[:, 0]) ** 2 + (dy / s[:, 1]) ** 2))   # (H,W,N)
    a = torch.sigmoid(op) * e
    color = torch.sigmoid(col)
    return ((a[..., None] * color[None, None]).sum(2) / (a.sum(2, keepdim=True) + 1e-6)).clamp(0, 1)

## 3 · Optimize — minimise reconstruction error

In [ ]:
opt = torch.optim.Adam([pos, logs, rot, col, op], 0.02); hist = []
for step in range(STEPS + 1):
    img = render(); loss = ((img - target) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        psnr = (-10 * torch.log10(loss)).item(); hist.append((step, psnr)); print(f"step {step:4d}  PSNR {psnr:5.2f} dB")

## 4 · Compare — target vs. splatted reconstruction + Gaussian centres

In [ ]:
with torch.no_grad(): img = render().cpu()
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
ax[0].imshow(target.cpu()); ax[0].set_title("target"); ax[0].axis("off")
ax[1].imshow(img);          ax[1].set_title("reconstruction"); ax[1].axis("off")
ax[2].plot(*zip(*hist), "-o"); ax[2].set_title("PSNR ↑"); ax[2].set_xlabel("step"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_gaussian_splatting_2d/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_gaussian_splatting_2d"; os.makedirs(run, exist_ok=True)
torch.save({k: v.detach().cpu() for k, v in {"pos": pos, "logs": logs, "rot": rot, "col": col, "op": op}.items()}, f"{run}/gaussians.pt")
json.dump({"psnr": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

### Where to go next
- Add/prune Gaussians by gradient magnitude (densification) — the trick that makes real 3DGS sharp.
- Lift to **3D**: Gaussians in space, splatted through a camera with depth-ordered alpha compositing → **[3D Gaussian Splatting](https://github.com/graphdeco-inria/gaussian-splatting)**.
- For real captures, use **[Nerfstudio splatfacto](https://docs.nerf.studio/)** on your own phone photos.